# NoSQL en Databricks · 00 · Entorno y fuentes JSON

**Objetivo:** preparar documentos de pedidos, clientes y productos de una plataforma ficticia.
Usaremos el modelo documental y lo procesaremos en tablas Delta con `VARIANT`.
JSON es un formato; NoSQL comprende varios modelos de bases de datos. Este laboratorio cubre
el modelo documental y el análisis, y no demuestra un motor NoSQL, índices, sharding ni consistencia distribuida.

Duración de este notebook: 20 minutos. Completá antes la guía de Databricks Free Edition.
No requiere ejecutar las clases 1 o 2 ni instalar bibliotecas.

## 1. Identidad y comprobación del entorno
Elegí el mismo `student_id` en los cuatro notebooks. `test` genera 12 pedidos aceptables;
`small`, 1.200. Esta escala es independiente de la utilizada en otras prácticas.

**Reejecución:** este notebook reconstruye las cinco tablas `nosql_*` de entrada y las fuentes
de `landing/nosql/` del alumno. Si modificaste esos datos, exportá tus cambios antes de repetirlo.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "test", ["test", "small"], "02 - Escala NoSQL")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_nosql_data

In [ ]:
config = CourseConfig(
    spark.sql("SELECT current_catalog()").first()[0],
    dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"),
)
repetitions = NOSQL_BLOCKS[config.scale]
try:
    probe = spark.sql("""
    SELECT typeof(parse_json('{"a":[1]}')) AS tipo,
           try_variant_get(parse_json('{"n":"2"}'), '$.n', 'int') AS numero,
           is_variant_null(parse_json('null')) AS null_json
    """).first()
    assert probe.tipo.lower() == "variant" and probe.numero == 2 and probe.null_json
    assert spark.sql("SELECT COUNT(*) FROM variant_explode(parse_json('[1,2]'))").first()[0] == 2
except Exception as error:
    raise RuntimeError(
        "El compute debe soportar VARIANT y variant_explode. Usá serverless actualizado "
        "o Databricks Runtime 15.4+. No instales paquetes para resolver este control."
    ) from error
create_course_namespace(spark, config)
print(f"Espacio: {config.catalog}.{config.schema}; escala: {config.scale}")

## 2. Crear fuentes JSON Lines en un volumen
Cada línea representa un documento completo, sin saltos internos. Escribirlos como texto permite
conservar también las entradas defectuosas. No usamos un array envolvente ni JSON multilínea.

Los pedidos incluyen `customer_snapshot`, `shipping`, `payment`, `risk` y una lista `items`.
También contienen `customer_id` y `product_id` como referencias lógicas.

In [ ]:
from pyspark.sql import functions as F
sources = build_nosql_sources(config.scale)
paths = {name: f"{config.volume_path}/nosql/{name}" for name in sources}
for name, lines in sources.items():
    spark.createDataFrame([(line,) for line in lines], "raw_json STRING")             .coalesce(1).write.mode("overwrite").text(paths[name])
print({name: len(lines) for name, lines in sources.items()})
import json
print(json.dumps(json.loads(sources["orders"][0]), ensure_ascii=False, indent=2))

## 3. Ingestar sin perder el original
Conservamos `raw_json`, ruta y momento de ingesta. `try_parse_json` convierte a `VARIANT` y
retorna NULL SQL si el texto es inválido. Un JSON `null` válido tiene otro significado.
`VARIANT` preserva la estructura lógica; puede cambiar el orden de claves y la representación numérica.
El texto original permite auditar el contenido exacto.

In [ ]:
raw = (spark.read.text(paths["orders"])
       .selectExpr("value AS raw_json", "_metadata.file_path AS source_file")
       .withColumn("doc", F.expr("try_parse_json(raw_json)"))
       .withColumn("ingested_at", F.current_timestamp()))
raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true")         .saveAsTable(qualified_table(config, "nosql_raw_orders"))
raw = spark.table(qualified_table(config, "nosql_raw_orders"))
display(raw.select("raw_json", "doc").limit(6))

## 4. Un esquema flexible también necesita contratos
Aceptamos un objeto con identificador y referencia de cliente no vacíos, `items` como array,
total convertible a decimal no negativo, moneda USD y estado conocido.
Se asigna el primer motivo de rechazo según el orden del CASE.

Este contrato inicial no valida todos los campos internos ni reconcilia el total con las líneas;
lo haremos con los datos aceptados en el notebook 02. Pensá qué reglas agregarías para producción.

In [ ]:
classified = raw.withColumn("quality_reason", F.expr("""
  CASE
    WHEN doc IS NULL THEN 'JSON_INVALIDO'
    WHEN is_variant_null(doc) THEN 'JSON_NULO'
    WHEN schema_of_variant(doc) NOT LIKE 'OBJECT<%' THEN 'DOCUMENTO_NO_OBJETO'
    WHEN COALESCE(try_variant_get(doc, '$.order_id', 'string'), '') = '' THEN 'ID_AUSENTE'
    WHEN COALESCE(try_variant_get(doc, '$.customer_id', 'string'), '') = '' THEN 'CLIENTE_AUSENTE'
    WHEN COALESCE(schema_of_variant(doc:items), '') NOT LIKE 'ARRAY<%' THEN 'ITEMS_NO_ARRAY'
    WHEN try_variant_get(doc, '$.total', 'decimal(12,2)') IS NULL THEN 'TOTAL_NO_NUMERICO'
    WHEN try_variant_get(doc, '$.total', 'decimal(12,2)') < 0 THEN 'TOTAL_NEGATIVO'
    WHEN COALESCE(try_variant_get(doc, '$.currency', 'string'), '') <> 'USD' THEN 'MONEDA_INVALIDA'
    WHEN COALESCE(try_variant_get(doc, '$.status', 'string'), '') NOT IN ('paid', 'pending', 'cancelled')
      THEN 'ESTADO_INVALIDO'
    ELSE NULL
  END
"""))
quarantine = classified.where("quality_reason IS NOT NULL")
orders = (classified.where("quality_reason IS NULL")
          .selectExpr("try_variant_get(doc, '$.order_id', 'string') AS order_id",
                      "raw_json", "doc", "source_file", "ingested_at"))
for name, dataframe in [("nosql_quarantine", quarantine), ("nosql_orders", orders)]:
    dataframe.write.format("delta").mode("overwrite").option("overwriteSchema", "true")             .saveAsTable(qualified_table(config, name))
for name, source_name, key in [("nosql_customers", "customers", "customer_id"),
                               ("nosql_products", "products", "product_id")]:
    reference = (spark.read.text(paths[source_name])
                 .selectExpr("value AS raw_json", "_metadata.file_path AS source_file")
                 .withColumn("doc", F.expr("parse_json(raw_json)"))
                 .withColumn(key, F.expr(f"try_variant_get(doc, '$.{key}', 'string')")))
    reference.write.format("delta").mode("overwrite").option("overwriteSchema", "true")             .saveAsTable(qualified_table(config, name))
display(spark.table("nosql_quarantine").select("quality_reason", "raw_json"))

## 5. Checkpoint de ingesta
Las aserciones comprueban conservación de filas, unicidad y los cuatro rechazos esperados.
En `test`: 16 entradas = 12 aceptadas + 4 rechazadas, 4 clientes y 6 productos.

In [ ]:
counts = {name: spark.table(name).count() for name in [
    "nosql_raw_orders", "nosql_orders", "nosql_quarantine", "nosql_customers", "nosql_products"]}
assert counts["nosql_raw_orders"] == 16 * repetitions
assert counts["nosql_orders"] == 12 * repetitions
assert counts["nosql_quarantine"] == 4 * repetitions
assert counts["nosql_raw_orders"] == counts["nosql_orders"] + counts["nosql_quarantine"]
assert counts["nosql_customers"] == 4 * repetitions and counts["nosql_products"] == 6
assert spark.sql("SELECT COUNT(DISTINCT order_id) FROM nosql_orders").first()[0] == counts["nosql_orders"]
reasons = {row.quality_reason: row['count'] for row in
           spark.table("nosql_quarantine").groupBy("quality_reason").count().collect()}
assert reasons == {name: repetitions for name in
                   ["JSON_INVALIDO", "JSON_NULO", "DOCUMENTO_NO_OBJETO", "TOTAL_NO_NUMERICO"]}
display(spark.createDataFrame(list(counts.items()), ["tabla", "filas"]))
print("Ingesta validada. Continuá con 01_modelado_documental.")

### Para responder
1. ¿Por qué leer primero como texto si finalmente queremos datos JSON?
2. ¿Qué diferencia hay entre JSON inválido, JSON `null` y un array JSON válido?
3. ¿Qué responsabilidad tiene el productor del documento y cuál el consumidor?
4. ¿Por qué `customer_id` no implica por sí mismo que exista el cliente referenciado?